# CKD EDA — read-only profiling notebook

**Read-only contract (Pattern N1):** this notebook observes the raw CKD data through the existing pipeline functions only.
It imports `load_raw_data`, `summarize`, and `clean_raw` from `src.*` and never re-implements quirk handling
(`?` to NaN, whitespace stripping, dtype coercion). It never fills, imputes, encodes, or fits anything —
working copies are named `eda_df` / `plot_df` and are only filtered or coerced for plotting.
Every missingness number shown here comes from `summarize()`, the single source of truth.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")  # kernel cwd is notebooks/ -> repo root (Pattern N1 shim)

ROOT = Path("..").resolve() if Path("..", "src").exists() else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ingestion.load_data import load_raw_data, summarize
from src.preprocessing.preprocess import NUMERIC_COLS, CATEGORICAL_COLS, clean_raw

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
eda_df = load_raw_data()  # configured raw path; quirks already normalized upstream

summary = summarize(eda_df)  # single source of truth for missingness numbers
summary

In [ ]:
missing_pct = eda_df.isna().mean().mul(100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 10))
missing_pct.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("missing %")
ax.set_title("Per-column missingness (raw CKD data)")
fig.savefig(FIG_DIR / "missingness_bar.png", bbox_inches="tight")
plt.close(fig)
missing_pct

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(eda_df.isna(), cbar_kws={"label": "missing"}, yticklabels=False, ax=ax)
ax.set_title("Missing-value map (yellow = missing)")
fig.savefig(FIG_DIR / "missingness_heatmap.png", bbox_inches="tight")
plt.close(fig)